In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

CATALOG, SCHEMA, VOLUME = "workspace", "default", "opssat-ad"
T_SILVER = f"{CATALOG}.{SCHEMA}.silver_telemetry"
T_GOLD = f"{CATALOG}.{SCHEMA}.gold_segment_features"
REFERENCE_FILE = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/dataset.csv"

s = spark.table(T_SILVER)

In [0]:
w = Window.partitionBy("segment").orderBy("ts")

t = (
    s.withColumn("t_sec", F.col("ts").cast("double"))
    .withColumn("gap", F.col("t_sec") - F.lag("t_sec").over(w))
    .withColumn("diff", F.col("value") - F.lag("value").over(w))
    .withColumn("diff2", F.col("diff") - F.lag("diff").over(w))
    .withColumn("smooth10", F.avg("value").over(w.rowsBetween(-9, 0)))
    .withColumn("smooth20", F.avg("value").over(w.rowsBetween(-19, 0)))
)

In [0]:
def is_peak(col):
    return ((F.col(col) > F.lag(col).over(w)) &
            (F.col(col) > F.lead(col).over(w))).cast("int")

t = (
    t.withColumn("pk", is_peak("value"))
    .withColumn("pk_s10", is_peak("smooth10"))
    .withColumn("pk_s20", is_peak("smooth20"))
    .withColumn("pk_d1", is_peak("diff"))
    .withColumn("pk_d2", is_peak("diff2"))
)

In [0]:
gold = (
    t.groupBy("segment", "channel", "anomaly", "train", "sampling")
    .agg(
        (F.max("t_sec") - F.min("t_sec")).alias("duration"),
        F.count("*").alias("len"),
        F.avg("value").alias("mean"),
        F.var_pop("value").alias("var"),
        F.stddev_pop("value").alias("std"),
        F.kurtosis("value").alias("kurtosis"),
        F.skewness("value").alias("skew"),
        F.sum("pk").alias("n_peaks"),
        F.sum("pk_s10").alias("smooth10_n_peaks"),
        F.sum("pk_s20").alias("smooth20_n_peaks"),
        F.sum("pk_d1").alias("diff_peaks"),
        F.sum("pk_d2").alias("diff2_peaks"),
        F.var_pop("diff").alias("diff_var"),
        F.var_pop("diff2").alias("diff2_var"),
        F.sum(F.col("gap") * F.col("gap")).alias("gaps_squared"),
    )
    .withColumn("len_weighted", F.col("len") * F.col("sampling"))
    .withColumn("var_div_duration", F.col("var") / F.col("duration"))
    .withColumn("var_div_len", F.col("var") / F.col("len"))
    .fillna(0)
)

gold.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(T_GOLD)
g = spark.table(T_GOLD)
print(f"Gold : {g.count():,} segments")
display(g.limit(10))

In [0]:
ref = spark.read.option("header", True).option("inferSchema", True).csv(REFERENCE_FILE)

FEATURES = ["duration", "len", "mean", "var", "std", "kurtosis", "skew",
            "n_peaks", "smooth10_n_peaks", "smooth20_n_peaks", "diff_peaks", "diff2_peaks",
            "diff_var", "diff2_var", "gaps_squared", "len_weighted",
            "var_div_duration", "var_div_len"]

j = g.alias("g").join(ref.alias("r"), "segment")
ecarts = j.select(*[
    F.percentile_approx(
        F.abs(F.col(f"g.{c}") - F.col(f"r.{c}")) / (F.abs(F.col(f"r.{c}")) + F.lit(1e-12)), 0.5
    ).alias(c) for c in FEATURES
]).first().asDict()

display(spark.createDataFrame([(k, float(v)) for k, v in ecarts.items()],
                              ["feature", "ecart_relatif_median"]))